In [7]:
import ssl
import urllib.request
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

In [8]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.flatten(1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [10]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

os.makedirs('./data', exist_ok=True)

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2 if os.name == 'posix' else 0)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2 if os.name == 'posix' else 0)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
verify_model = TinyCNN().to(device)
dummy_input = torch.randn(1, 3, 32, 32).to(device)

print("--- Spatial Dimension Verification Run ---")
print("Input Shape:", dummy_input.shape)
x = verify_model.pool(F.relu(verify_model.conv1(dummy_input)))
print("After Conv1 + Pool:", x.shape)
x = verify_model.pool(F.relu(verify_model.conv2(x)))
print("After Conv2 + Pool:", x.shape)
x = verify_model.pool(F.relu(verify_model.conv3(x)))
print("After Conv3 + Pool:", x.shape)
x = x.flatten(1)
print("After Flattening Layer:", x.shape)
print("------------------------------------------")

--- Spatial Dimension Verification Run ---
Input Shape: torch.Size([1, 3, 32, 32])
After Conv1 + Pool: torch.Size([1, 32, 16, 16])
After Conv2 + Pool: torch.Size([1, 64, 8, 8])
After Conv3 + Pool: torch.Size([1, 128, 4, 4])
After Flattening Layer: torch.Size([1, 2048])
------------------------------------------


In [12]:
model = TinyCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

train_accuracies = []
val_accuracies = []

for epoch in range(10):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
        
    train_acc = 100.0 * correct_train / total_train
    train_accuracies.append(train_acc)
    
    model.eval()
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
            
    val_acc = 100.0 * correct_val / total_val
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch+1}/10 | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

Epoch 1/10 | Loss: 1.5665 | Train Acc: 42.49% | Val Acc: 58.49%
Epoch 2/10 | Loss: 1.1753 | Train Acc: 57.93% | Val Acc: 65.74%
Epoch 3/10 | Loss: 1.0093 | Train Acc: 64.47% | Val Acc: 70.34%
Epoch 4/10 | Loss: 0.9106 | Train Acc: 68.02% | Val Acc: 71.44%
Epoch 5/10 | Loss: 0.8521 | Train Acc: 70.20% | Val Acc: 74.29%
Epoch 6/10 | Loss: 0.8020 | Train Acc: 72.10% | Val Acc: 75.37%
Epoch 7/10 | Loss: 0.7739 | Train Acc: 73.28% | Val Acc: 75.55%
Epoch 8/10 | Loss: 0.7430 | Train Acc: 74.34% | Val Acc: 76.26%
Epoch 9/10 | Loss: 0.7203 | Train Acc: 75.02% | Val Acc: 77.12%
Epoch 10/10 | Loss: 0.6913 | Train Acc: 75.98% | Val Acc: 75.84%


In [13]:
torch.save(torch.tensor(val_accuracies), "cnn_val_accs.pt")

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), train_accuracies, label='Training Accuracy', marker='o')
plt.plot(range(1, 11), val_accuracies, label='Validation Accuracy', marker='s')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('TinyCNN Baseline Performance')
plt.legend()
plt.grid(True)
plt.savefig('cnn_baseline_plot.png')
plt.close()
print("Saved performance graph: cnn_baseline_plot.png")
print("Saved baseline accuracy weights: cnn_val_accs.pt")

Saved performance graph: cnn_baseline_plot.png
Saved baseline accuracy weights: cnn_val_accs.pt
